In [38]:
import os

In [39]:
print(os.getcwd())

/workspaces/ml-zoomcamp/05-deployment/homework


In [5]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 65.7 MB/s  0:00:00s eta 0:00:01

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /home/codespace/.local/bin
  uv
  uvx
everything's installed!


In [7]:
!uv sync --locked

Resolved 19 packages in 2ms
Checked 18 packages in 26ms


In [8]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [9]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


# Question 1 — environment check (0.13.5)

In [10]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


# Question 2 — locked dependency (1.7.2)

In [11]:
!grep scikit pyproject.toml

    "scikit-learn==1.7.2",


In [12]:
!grep scikit uv.lock

    { name = "scikit-learn" },
    { name = "scikit-learn", specifier = "==1.7.2" },
name = "scikit-learn"
sdist = { url = "https://files.pythonhosted.org/packages/98/c2/a7855e41c9d285dfe86dc50b250978105dce513d6e459ea66a6aeb0e1e0c/scikit_learn-1.7.2.tar.gz", hash = "sha256:20e9e49ecd130598f1ca38a1d85090e1a600147b9c02fa6f15d69cb53d968fda", size = 7193136, upload-time = "2025-09-09T08:21:29.075Z" }
    { url = "https://files.pythonhosted.org/packages/43/83/564e141eef908a5863a54da8ca342a137f45a0bfb71d1d79704c9894c9d1/scikit_learn-1.7.2-cp311-cp311-macosx_10_9_x86_64.whl", hash = "sha256:c7509693451651cd7361d30ce4e86a1347493554f172b1c72a39300fa2aea79e", size = 9331967, upload-time = "2025-09-09T08:20:32.421Z" },
    { url = "https://files.pythonhosted.org/packages/18/d6/ba863a4171ac9d7314c4d3fc251f015704a2caeee41ced89f321c049ed83/scikit_learn-1.7.2-cp311-cp311-macosx_12_0_arm64.whl", hash = "sha256:0486c8f827c2e7b64837c731c8feff72c0bd2b998067a8a9cbc10643c31f0fe1", size = 8648645, upload-ti

# Question 3 — load the model (0.533)

In [14]:
import pickle

model_file = 'pipeline.bin'

with open(model_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)

lead = {
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}

def predict():
    X = dv.transform([lead])
    y_pred = model.predict_proba(X)[0, 1]
    converted = y_pred >= 0.5

    result = {
        'conversion_probability': round(float(y_pred),3),
        'converted': bool(converted)
    }

    return result


if __name__ == "__main__":
    print(predict())
    

{'conversion_probability': 0.533, 'converted': True}


# Question 4 — serve the model (0.77)

In [20]:
import requests

In [21]:
url = 'http://127.0.0.1:9696/predict'

In [22]:
lead = {
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}

In [23]:
response = requests.post(url, json=lead).json()

In [24]:
response

{'conversion_probability': 0.77, 'conversion': True}

# Container

In [25]:
!docker build -t zoomcamp-model:2026-hw5 .



[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.1s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.0s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.0s
[+] Building 0.3s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.2s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.2s
[+] Building 0.3s (1/4)                                          docker:default
 => [internal] load build definition f

In [33]:
!docker images

                                                            i Info →   U  In Use
IMAGE                     ID             DISK USAGE   CONTENT SIZE   EXTRA
zoomcamp-model:2026-hw5   95e3d1d9447e        961MB          227MB        


In [35]:
!docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5

INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
INFO:     172.17.0.1:46516 - "POST /predict HTTP/1.1" 200 OK
INFO:     172.17.0.1:55008 - "GET /health HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1]


# Question 5 — look at the container configuration (python:3.11.15-slim-bookworm)

In [34]:
!docker run --rm zoomcamp-model:2026-hw5 python --version

Running uvicorn 0.38.0 with CPython 3.11.15 on Linux


# Question 6 — run the container
### {'conversion_probability': 0.77, 'conversion': True}

### {"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}


In [40]:
!grep model_sha256 model_metadata.json

  "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1",
